## **Aim**
To implement a program that monitors a specified folder and records file creation, modification, deletion, and access events in real-time.

## **Algorithm**
**Step 1:** Import the `watchdog` library components: `Observer`, `FileSystemEventHandler`, and `time`.

**Step 2:** Define a custom event handler class inheriting from `FileSystemEventHandler` to override `on_created`, `on_modified`, `on_deleted`, and `on_moved` methods.

**Step 3:** In each event method, capture the current timestamp and event details (file path, event type).

**Step 4:** Print or log the event information to the console in a structured format.

**Step 5:** Create an `Observer` instance, schedule it to watch the target directory recursively.

**Step 6:** Start the observer and keep the program running until interrupted (Ctrl+C).

**Step 7:** On keyboard interrupt, stop the observer gracefully.

In [1]:
import time
import os
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
from datetime import datetime

class FolderMonitor(FileSystemEventHandler):
    def __init__(self):
        super().__init__()
    
    def log_event(self, event_type, src_path, dest_path=None):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        if dest_path:
            print(f"[{timestamp}] {event_type}: {src_path} -> {dest_path}")
        else:
            print(f"[{timestamp}] {event_type}: {src_path}")
    
    def on_created(self, event):
        if not event.is_directory:
            self.log_event("CREATED", event.src_path)
    
    def on_modified(self, event):
        if not event.is_directory:
            self.log_event("MODIFIED", event.src_path)
    
    def on_deleted(self, event):
        if not event.is_directory:
            self.log_event("DELETED", event.src_path)
    
    def on_moved(self, event):
        if not event.is_directory:
            self.log_event("RENAMED/MOVED", event.src_path, event.dest_path)

def main():
    watch_path = "./monitor_test"
    os.makedirs(watch_path, exist_ok=True)
    
    print(f"Monitoring folder: {watch_path}")
    print("Press Ctrl+C to stop monitoring...\n")
    
    event_handler = FolderMonitor()
    observer = Observer()
    observer.schedule(event_handler, watch_path, recursive=True)
    observer.start()
    
    try:
        # Simulate some file events for demonstration
        time.sleep(1)
        with open(os.path.join(watch_path, "test_file.txt"), "w") as f:
            f.write("Initial content")
        time.sleep(1)
        with open(os.path.join(watch_path, "test_file.txt"), "a") as f:
            f.write("\nAppended content")
        time.sleep(1)
        os.rename(os.path.join(watch_path, "test_file.txt"), os.path.join(watch_path, "renamed_file.txt"))
        time.sleep(1)
        os.remove(os.path.join(watch_path, "renamed_file.txt"))
        time.sleep(2)
    except KeyboardInterrupt:
        pass
    finally:
        observer.stop()
        observer.join()
        print("\nMonitoring stopped.")

if __name__ == "__main__":
    main()

Monitoring folder: ./monitor_test
Press Ctrl+C to stop monitoring...



[2026-08-20 08:57:32] CREATED: ./monitor_test/test_file.txt
[2026-08-20 08:57:32] MODIFIED: ./monitor_test/test_file.txt


[2026-08-20 08:57:33] MODIFIED: ./monitor_test/test_file.txt


[2026-08-20 08:57:34] RENAMED/MOVED: ./monitor_test/test_file.txt -> ./monitor_test/renamed_file.txt


[2026-08-20 08:57:35] DELETED: ./monitor_test/renamed_file.txt



Monitoring stopped.


## **Result**
This the program successfully monitors a specified folder and records file creation, modification, deletion, and access events.